## **ReAct: Build Reasoning and Acting AI Agents with LangGraph**

#### 1. Web Search Tool
### Tavily Search API Key Setup

In [11]:
import warnings 
warnings.filterwarnings('ignore')

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.tools import tool
import os
import json

#tavily_api_key = os.environ.get("TAVILY_API_KEY")
from dotenv import load_dotenv
load_dotenv()

# Initialize the Tavily search tool
search = TavilySearchResults()

@tool
def search_tool(query: str):
    """
    Search the web for information using Tavily API.

    :param query: The search query string
    :return: Search results related to the query
    """
    return search.invoke(query)

### Testing the Search Tool

In [6]:
search_tool.invoke("What's the weather like in Tokyo today?")

[{'title': 'Tokyo Weather Forecast July 11 2026 - Facebook',
  'url': 'https://www.facebook.com/fb-answers/tokyo-weather-forecast-july-11-2026',
  'content': 'Tokyo is expecting rain and thunderstorms on July 11, 2026, with up to 150mm of rain possible in the Kanto region. Up to 150 millimeters of rain could fall in',
  'score': 0.8282873},
 {'title': 'Japan Weather in September 2026: Heat Lingers, Typhoons',
  'url': 'https://www.agatetravel.com/japan/weather-in-september.html',
  'content': 'Moving northward to central Japan cities like Tokyo and Kyoto, temperatures begin to show a slight downward trend this month. The beginning and middle parts of the month often remain somewhat hot, with daytime temperatures around 25-30°C (77-86°F). Occasionally, the peaks may still reach around 35°C (95°F). By the last few days of the month, temperatures typically drop to around 20-25°C (68-77°F), making the weather more comfortable even for hiking. [...] September marks the onset of autumn in Ja

#### 2. Clothing Recommendation Tool

In [7]:
@tool
def recommend_clothing(weather: str) -> str:
    """
    Returns a clothing recommendation based on the provided weather description.

    This function examines the input string for specific keywords or temperature indicators 
    (e.g., "snow", "freezing", "rain", "85°F") to suggest appropriate attire. It handles 
    common weather conditions like snow, rain, heat, and cold by providing simple and practical 
    clothing advice.

    :param weather: A brief description of the weather (e.g., "Overcast, 64.9°F")
    :return: A string with clothing recommendations suitable for the weather
    """
    weather = weather.lower()
    if "snow" in weather or "freezing" in weather:
        return "Wear a heavy coat, gloves, and boots."
    elif "rain" in weather or "wet" in weather:
        return "Bring a raincoat and waterproof shoes."
    elif "hot" in weather or "85" in weather:
        return "T-shirt, shorts, and sunscreen recommended."
    elif "cold" in weather or "50" in weather:
        return "Wear a warm jacket or sweater."
    else:
        return "A light jacket should be fine."

#### Creating the Tool Registry

In [8]:
tools=[search_tool,recommend_clothing]

tools_by_name={ tool.name:tool for tool in tools}

## Setting Up the Language Model

### Initializing the AI Model

In [13]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

openai_api_key = os.getenv("OPENAI_KEY")

model = ChatOpenAI(model="gpt-4o-mini", api_key=openai_api_key)

### Creating the System Prompt

In [14]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage,SystemMessage

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a helpful AI assistant that thinks step-by-step and uses tools when needed.

When responding to queries:
1. First, think about what information you need
2. Use available tools if you need current data or specific capabilities  
3. Provide clear, helpful responses based on your reasoning and any tool results

Always explain your thinking process to help users understand your approach.
"""),
    MessagesPlaceholder(variable_name="scratch_pad")
])

### Binding Tools to the Model

In [15]:
model_react=chat_prompt|model.bind_tools(tools)

## Setting Agent State

The agent must maintain context across multiple reasoning and acting steps

In [17]:
from typing import (Annotated,Sequence,TypedDict)
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    """The state of the agent."""

    # add_messages is a reducer
    # Link for more details: https://langchain-ai.github.io/langgraph/concepts/low_level/#reducers
    messages: Annotated[Sequence[BaseMessage], add_messages]

**Key Concepts:**
- **State**: Contains the conversation history and context.
- **Reducer**: `add_messages` automatically handles adding new messages to the conversation.
- **Type Safety**: TypedDict ensures our state structure is well-defined.

### Demonstrating State Management

In [18]:
state: AgentState = {"messages": []}

# append a message using the reducer properly
state["messages"] = add_messages(state["messages"], [HumanMessage(content="Hi")])
print("After greeting:", state["messages"])

# add another message (e.g. a question)
state["messages"] = add_messages(state["messages"], [HumanMessage(content="Weather in NYC?")])
print("After question:", state)

After greeting: [HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='e53a5266-91c8-4c5b-8ea1-6dfb2690a6ea')]
After question: {'messages': [HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='e53a5266-91c8-4c5b-8ea1-6dfb2690a6ea'), HumanMessage(content='Weather in NYC?', additional_kwargs={}, response_metadata={}, id='01f23296-f3eb-4c42-86e3-4777a833d2ba')]}


This demonstrates how the state accumulates context over the conversation.

## Manual ReAct Execution

Manually steping through a ReAct cycle:

### Step 1: Initial Query Processing

In [ ]:
dummy_state: AgentState = {
    # The user asks a complex question requiring current data.
    "messages": [HumanMessage( "What's the weather like in Zurich, and what should I wear based on the temperature?")]}

# The model analyzes the query and realizes it needs to search for weather information.
# The model generates a tool call for the search.
response = model_react.invoke({"scratch_pad":dummy_state["messages"]})

dummy_state["messages"]=add_messages(dummy_state["messages"],[response])

### Step 2: Tool Execution

In [ ]:
# Extract the tool call from the model's response.
tool_call = response.tool_calls[-1]
print("Tool call:", tool_call)

# Execute the tool using the specified arguments.
tool_result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
print("Tool result preview:", tool_result[0]['title'])

# Create a ToolMessage containing the results.
tool_message = ToolMessage(
    content=json.dumps(tool_result),
    name=tool_call["name"],
    tool_call_id=tool_call["id"]
)

# Add the tool result to the conversation state.
dummy_state["messages"] = add_messages(dummy_state["messages"], [tool_message])

Tool call: {'name': 'search_tool', 'args': {'query': 'current weather in Zurich'}, 'id': 'call_RASQXVHBx6WPO5dZ4qeqPX7I', 'type': 'tool_call'}
Tool result preview: Zurich weather today & 14 days — September 2026 | 14cast.com


### Step 3: Processing Results and Next Action

In [ ]:
# The model processes the search results.
response = model_react.invoke({"scratch_pad": dummy_state["messages"]})
dummy_state['messages'] = add_messages(dummy_state['messages'], [response])

# check if the model wants to use another tool
if response.tool_calls:
    tool_call = response.tool_calls[0]
    tool_result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
    tool_message = ToolMessage(
        content=json.dumps(tool_result),
        name=tool_call["name"],
        tool_call_id=tool_call["id"]
    )
    dummy_state['messages'] = add_messages(dummy_state['messages'], [tool_message])

### Step 4: Final Response Generation

In [ ]:
response = model_react.invoke({"scratch_pad": dummy_state["messages"]})
print("Final response generated:", response.content is not None)
print("More tools needed:", bool(response.tool_calls))